# 11 — Debentures, internal read

**The question:** *What does ANBIMA say about a debenture (indexer, spread,
schedule, indicative rate), and which CVM offering issued it?*

> **Internal only.** This notebook reads private tables. The ANBIMA permission
> covers collection and storage, not redistribution. These rows are not in
> schema `api`, `serve/`, the MCP or the dashboard, and the public key cannot
> read them. Do not export, publish or share the output of this notebook.

This notebook does not use the public API (`SiloClient`). It reads the
warehouse directly with the owner's `POSTGRES_URL`, through
`src.store.pg_client`, the one connection path the repository allows. Every
query runs in a `READ ONLY` transaction, so the notebook cannot change data.

The tables:

| Table | Holds | Key |
| --- | --- | --- |
| `anbima_debenture` | characteristics: issuer, indexer, spread, dates | `codigo` |
| `anbima_debenture_event` | the event agenda: interest, amortisation, maturity | `codigo, data_evento, evento` |
| `anbima_debenture_indicative` | ANBIMA indicative rate and PU, the last 5 sessions of each capture | `codigo, data_referencia` |
| `anbima_debenture_capture` | one row for each page read, with its status | `capture_id, codigo, page` |
| `cvm_oferta_distrib_160` | every Resolução CVM 160 offering (2023 on) | `numero_requerimento` |
| `cvm_oferta_distrib_400` | registered ICVM 400 debenture offerings | `numero_registro_oferta, serie` |


## Setup

Run Jupyter from the repository root with the pipeline's own requirements
(`pip install -r requirements.txt`), and export the owner's connection string
first:

```bash
export POSTGRES_URL="postgresql://...?sslmode=require"   # never commit it
jupyter lab notebooks/
```


In [ ]:
import os
import sys
from pathlib import Path

# The notebook runs from notebooks/; the repository root holds src/.
ROOT = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import pandas as pd

from src.store.pg_client import get_pg_client

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

pg = get_pg_client()  # raises when POSTGRES_URL is not set


def q(sql: str, params=None) -> pd.DataFrame:
    """Run one SELECT in a READ ONLY transaction and return a DataFrame."""
    with pg.cursor() as cur:
        cur.execute("BEGIN READ ONLY")
        try:
            cur.execute("SET LOCAL statement_timeout = '60s'")
            cur.execute(sql, params)
            cols = [c.name for c in cur.description]
            rows = cur.fetchall()
        finally:
            cur.execute("ROLLBACK")
    return pd.DataFrame(rows, columns=cols)

## The first read is always the audit log

Every capture and every load writes one `cvm_ingest_log` row. Read it first:
it tells you when each source last loaded, and whether that run was complete.
A run with `status = 'error'` stored only part of its pages. Its `error_msg`
says how many pages failed.


In [ ]:
q("""
    SELECT DISTINCT ON (entity, doc_type)
           entity, doc_type, status, rows_upserted,
           started_at AT TIME ZONE 'America/Sao_Paulo' AS started_brt,
           left(error_msg, 120) AS error_msg
      FROM cvm_ingest_log
     WHERE entity IN ('anbima', 'oferta') AND doc_type IN ('debentures', 'distrib_160', 'distrib')
     ORDER BY entity, doc_type, started_at DESC
""")

## How the ANBIMA data gets here

The ANBIMA tables fill only when someone runs the capture. It is not in the
daily run yet. It opens data.anbima.com.br in a headless Chromium (Playwright)
and reads three pages for each code. Run it from a terminal, not from this
notebook:

```bash
pip install -r requirements.txt && playwright install chromium
python -m src.pipeline.anbima_debenture_pipeline --codes CGOSA2,AESLA8            # store
python -m src.pipeline.anbima_debenture_pipeline --codes CGOSA2 --dry-run          # no database
python -m src.pipeline.anbima_debenture_pipeline --codes CGOSA2 --pages precos     # rates only
```

The CVM offerings come from the backfill workflow (`backfill.yml`, entity
`oferta`) or from `python -m src.pipeline.run_backfill --entity oferta --cvm-only`.


## Capture health

One row for each page read. `no_data` is a page that loaded and has nothing:
a bond with no indicative rate shows no rate table. `fetch_error` and
`parse_error` are failures. A `parse_error` row keeps the raw page text, so a
change in ANBIMA's layout can be debugged.


In [ ]:
q("""
    SELECT page, status, count(*) AS pages, max(observed_at) AT TIME ZONE 'America/Sao_Paulo' AS last_read_brt
      FROM anbima_debenture_capture
     GROUP BY page, status
     ORDER BY page, status
""")

## Characteristics

`remuneracao` is the text ANBIMA publishes. The pipeline splits it into
`indexador`, `forma_remuneracao` and `taxa_remuneracao_pct`, and leaves them
NULL when it does not recognise the text. Read the rate with its form:

- `spread`: indexer plus a spread, for example IPCA + 6.57 % or DI + 0.49 %.
- `percentual`: a percent of the indexer, for example 114.65 % of DI.
- `prefixado`: a fixed rate.


In [ ]:
bonds = q("""
    SELECT codigo, isin, emissor, emissor_cnpj, emissao, serie,
           remuneracao, indexador, forma_remuneracao, taxa_remuneracao_pct,
           data_emissao, data_vencimento, lei_12431
      FROM anbima_debenture
     ORDER BY codigo
""")
bonds

In [ ]:
bonds.groupby(["indexador", "forma_remuneracao"], dropna=False).size().rename("bonds")

## Indicative rates

`taxa_indicativa_pct` is ANBIMA's model value, **not a trade**. For a
`percentual` (% DI) bond every rate on the page is a percent of DI, not an
annual rate. Each capture stores the last 5 sessions that ANBIMA shows, and a
later capture adds new sessions, so the history grows only forward. A day with
no row is a day with no capture: never fill it.


In [ ]:
CODE = "CGOSA2"  # change to any captured code

q("""
    SELECT i.data_referencia, i.taxa_indicativa_pct, i.taxa_compra_pct, i.taxa_venda_pct,
           i.pu_indicativo_brl, i.duration_du, d.forma_remuneracao
      FROM anbima_debenture_indicative i
      JOIN anbima_debenture d USING (codigo)
     WHERE i.codigo = %s
     ORDER BY i.data_referencia DESC
""", (CODE,))

## The event agenda

The agenda holds every interest, amortisation and maturity event as ANBIMA
publishes it. When ANBIMA drops an event, its old row stays with its old
`capture_id`. So read only the rows of the latest capture of each code.


In [ ]:
q("""
    WITH latest AS (
        SELECT DISTINCT ON (codigo) codigo, capture_id
          FROM anbima_debenture_capture
         WHERE page = 'agenda' AND status IN ('parsed', 'no_data')
         ORDER BY codigo, observed_at DESC
    )
    SELECT e.data_evento, e.evento, e.percentual_taxa_pct, e.valor_pago_brl, e.status
      FROM anbima_debenture_event e
      JOIN latest l USING (codigo, capture_id)
     WHERE e.codigo = %s AND e.data_evento >= CURRENT_DATE
     ORDER BY e.data_evento
     LIMIT 20
""", (CODE,))

## CVM offerings

Neither CVM file has an ISIN or a series, so a CVM offering does not join to a
traded code directly. The Resolução CVM 160 file starts in 2023. The ICVM 400
table holds only the registered debenture offerings; the ICVM 476 offerings
have no natural key and are not loaded.


In [ ]:
q("""
    SELECT date_trunc('year', data_registro)::date AS year,
           count(*) AS offerings,
           round(sum(valor_total_registrado) / 1e9, 1) AS registered_brl_bn
      FROM cvm_oferta_distrib_160
     WHERE valor_mobiliario = 'Debêntures' AND data_registro IS NOT NULL
     GROUP BY 1
     ORDER BY 1
""")

## Join ANBIMA to CVM

The join goes through the issuer and the issue number: ANBIMA's
`emissor_cnpj` and the leading number of `emissao` ("12ª Emissão" gives 12)
against CVM's `cnpj_emissor` and `emissao`. One CVM offering can cover several
series, so several codes can match one offering. A bond issued before 2023 under
ICVM 476 has no CVM row here: no match is the expected result, not an error.


In [ ]:
q("""
    SELECT d.codigo, d.emissor, d.emissao, d.serie, d.data_emissao,
           o.numero_requerimento, o.data_registro, o.status_requerimento,
           o.valor_total_registrado, o.publico_alvo, o.nome_lider
      FROM anbima_debenture d
      LEFT JOIN cvm_oferta_distrib_160 o
        ON o.cnpj_emissor = d.emissor_cnpj
       AND o.emissao = substring(d.emissao FROM '^\\d+')
       AND o.valor_mobiliario = 'Debêntures'
     ORDER BY o.numero_requerimento IS NULL, d.codigo
""")